
# 02 — SegFormer segmentation, spatial cross-validation and LSOA features

This notebook converts valid Google Street View images into reproducible neighbourhood-level semantic features and creates the final leakage-safe spatial evaluation design. It preserves image-level auditability, resumable inference, numerical and visual quality control, LSOA aggregation, compositional features, and production safeguards.


In [1]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

In [2]:
# Keep Colab's core stack pinned; upgrading it can break binary compatibility.
# %pip installs into the active kernel.

%pip install -q transformers accelerate pyarrow

!python -m pip check
%pip install -q jedi

In [3]:
!python -m pip check

In [4]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

## 1. Imports, reproducibility, paths, and configuration

In [5]:
from __future__ import annotations

from contextlib import nullcontext
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple
import gc
import hashlib
import json
import os
import platform
import random
import shutil
import time
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image, ImageFile, UnidentifiedImageError
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoImageProcessor, AutoModelForSemanticSegmentation

warnings.filterwarnings("ignore", category=FutureWarning)
ImageFile.LOAD_TRUNCATED_IMAGES = False

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

# Stable inference without disabling normal GPU optimisations.
torch.backends.cudnn.benchmark = True

PROJECT_DIR = Path("/content/drive/MyDrive/leeds_gsv_project")
OUTPUT_DIR = PROJECT_DIR / "outputs_10points_metadata_once"
IMAGE_DIR = OUTPUT_DIR / "gsv_images"
CHECKPOINT_DIR = OUTPUT_DIR / "segmentation_checkpoints"
QC_DIR = OUTPUT_DIR / "segmentation_qc"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
QC_DIR.mkdir(parents=True, exist_ok=True)

LSOA_ID_COL = "LSOA21CD"
IMAGE_METADATA_CSV = OUTPUT_DIR / "gsv_image_download_log_10points.csv"
SEG_IMAGE_FEATURES_CSV = OUTPUT_DIR / "segmentation_image_features_19class.csv"
SEG_LSOA_FEATURES_CSV = OUTPUT_DIR / "segmentation_lsoa_features_19class_mean_sd.csv"
SEG_FAILURES_CSV = OUTPUT_DIR / "segmentation_failures.csv"
SEG_RUN_SUMMARY_JSON = OUTPUT_DIR / "segmentation_run_summary.json"
SEG_CONFIG_JSON = OUTPUT_DIR / "segmentation_config.json"

MODEL_NAME = "nvidia/segformer-b0-finetuned-cityscapes-1024-1024"

CITYSCAPES_19 = [
    "road", "sidewalk", "building", "wall", "fence", "pole",
    "traffic_light", "traffic_sign", "vegetation", "terrain", "sky",
    "person", "rider", "car", "truck", "bus", "train", "motorcycle", "bicycle"
]

@dataclass(frozen=True)
class SegmentationConfig:
    batch_size: int = 4           # T4: 4; L4/A100: test 8
    num_workers: int = 2          # Drive I/O is usually the bottleneck; 2 is conservative
    checkpoint_every_batches: int = 50
    max_retry_attempts: int = 2
    use_amp: bool = True          # CUDA mixed precision
    amp_dtype: str = "float16"
    min_file_size_bytes: int = 10_000
    qc_sum_tolerance: float = 1e-4
    low_information_threshold: float = 0.90
    min_images_per_lsoa: int = 4
    preferred_images_per_lsoa: int = 8
    test_mode: bool = False
    test_n_images: int = 200
    force_reprocess_successes: bool = False
    use_local_image_cache: bool = False
    local_image_cache_dir: str = "/content/gsv_images"
    save_qc_masks: bool = True
    n_qc_masks: int = 24
    n_spatial_folds: int = 5
    n_spatial_blocks: int = 120
    rebuild_spatial_folds: bool = True
    spatial_block_linkage: str = "complete"
    clr_pseudocount: float = 1e-6
    minimum_fold_class_count: int = 20
    fold_optimizer_restarts: int = 80
    fold_optimizer_iterations: int = 20_000
    fold_weight_lsoas: float = 1.5
    fold_weight_images: float = 1.0
    fold_weight_classes: float = 4.0
    fold_weight_compactness: float = 0.35
    fold_weight_features: float = 1.25
    maximum_fold_size_cv: float = 0.12
    maximum_feature_smd: float = 0.75

CONFIG = SegmentationConfig()

with open(SEG_CONFIG_JSON, "w") as f:
    json.dump(asdict(CONFIG), f, indent=2)

print(json.dumps(asdict(CONFIG), indent=2))


## 2. Runtime and GPU checks

In [6]:
def get_runtime_report() -> Dict[str, Any]:
    report = {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "python": platform.python_version(),
        "torch": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
        "cuda_version": torch.version.cuda,
        "device_count": torch.cuda.device_count(),
    }
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        report.update({
            "gpu_name": torch.cuda.get_device_name(0),
            "gpu_memory_gb": round(props.total_memory / (1024**3), 2),
        })
    return report

runtime_report = get_runtime_report()
print(json.dumps(runtime_report, indent=2))

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU is not enabled. In Colab select Runtime → Change runtime type → GPU, then rerun."
    )

gpu_name = runtime_report["gpu_name"].lower()
if "t4" in gpu_name and CONFIG.batch_size > 4:
    print("Warning: for a T4, start with batch_size=4.")
elif any(name in gpu_name for name in ["l4", "a100"]) and CONFIG.batch_size < 8:
    print("This GPU may support batch_size=8, but batch_size=4 is the safer starting point.")

## 3. Load and audit downloaded image metadata


## Final spatial cross-validation design

Segmentation is performed independently of cross-validation. After successful images are aggregated to one row per LSOA, the notebook creates the **final five folds** using a Spatial+-inspired two-stage procedure:

1. **Spatial blocking:** complete-linkage agglomerative hierarchical clustering groups nearby LSOAs into many compact micro-blocks. Every LSOA belongs to exactly one block.
2. **Block-to-fold optimisation:** whole blocks are allocated to folds while jointly balancing LSOA counts, image coverage, deprivation classes, semantic feature distributions, and geographic compactness.

This order is deliberate. SegFormer is used only for fixed pretrained semantic-segmentation inference; its weights are not updated with project images or IMD labels. Notebook 04 fits each candidate's configured preprocessing and hyperparameters within the relevant training partition: median imputation for all candidates, CLR or ILR plus scaling for logistic regression, and class-weight options where supported.


In [7]:

# Preflight downloaded images; build folds after LSOA aggregation.

from IPython.display import display

BASE_REQUIRED_COLUMNS = {
    "request_id", "point_id", "image_path", "filename", LSOA_ID_COL,
    "download_status", "heading_angle", "IMDRank", "IMDDecile",
    "DeprivationClass", "ClassIdx", "sample_lat", "sample_lon",
}
VALID_DOWNLOAD_STATUSES = {"downloaded", "existing_valid", "skipped_existing_valid"}


def resolve_image_path(path_value: Any) -> Path:
    if pd.isna(path_value) or not str(path_value).strip():
        return Path("")
    original = Path(str(path_value).strip())
    return Path(CONFIG.local_image_cache_dir) / original.name if CONFIG.use_local_image_cache else original


def prepare_local_image_cache(image_paths: Sequence[str]) -> None:
    if not CONFIG.use_local_image_cache:
        return
    paths = [Path(str(p).strip()) for p in image_paths if pd.notna(p) and str(p).strip()]
    parents = sorted({p.parent for p in paths})
    if len(parents) != 1:
        raise ValueError("Local caching requires all source images to share one parent directory.")
    source, destination = parents[0], Path(CONFIG.local_image_cache_dir)
    if not source.exists():
        raise FileNotFoundError(f"Image source directory not found: {source}")
    destination.mkdir(parents=True, exist_ok=True)
    shutil.copytree(source, destination, dirs_exist_ok=True)


def inspect_image_file(path_value: Any) -> pd.Series:
    path = Path(str(path_value))
    exists = path.is_file()
    size = path.stat().st_size if exists else 0
    return pd.Series({"file_exists": exists, "file_size_bytes": size})


if not IMAGE_METADATA_CSV.exists():
    raise FileNotFoundError(f"Metadata file not found: {IMAGE_METADATA_CSV}")

image_meta_raw = pd.read_csv(IMAGE_METADATA_CSV, low_memory=False)
missing = BASE_REQUIRED_COLUMNS.difference(image_meta_raw.columns)
if missing:
    raise KeyError(f"Metadata is missing required columns: {sorted(missing)}")

image_meta_raw = image_meta_raw.copy()
image_meta_raw["_source_row"] = np.arange(len(image_meta_raw))
for col in ["request_id", "point_id", "filename", "image_path", LSOA_ID_COL,
            "DeprivationClass", "download_status"]:
    image_meta_raw[col] = image_meta_raw[col].astype("string").str.strip()

for col in ["sample_lat", "sample_lon", "IMDRank", "IMDDecile", "ClassIdx", "heading_angle"]:
    image_meta_raw[col] = pd.to_numeric(image_meta_raw[col], errors="coerce")

image_meta_raw["download_status_normalised"] = (
    image_meta_raw["download_status"].str.lower().str.strip()
)

# Check target consistency before GPU inference.
lsoa_target_check = image_meta_raw.groupby(LSOA_ID_COL)[
    ["IMDRank", "IMDDecile", "DeprivationClass", "ClassIdx"]
].nunique(dropna=False)
conflicting_lsoas = lsoa_target_check.loc[(lsoa_target_check > 1).any(axis=1)]
if not conflicting_lsoas.empty:
    raise ValueError(
        f"{len(conflicting_lsoas)} LSOAs contain conflicting deprivation metadata. "
        "Correct the download log before the final run."
    )

required_nonmissing = [LSOA_ID_COL, "request_id", "point_id", "image_path",
                       "sample_lat", "sample_lon", "DeprivationClass", "ClassIdx"]
if image_meta_raw[required_nonmissing].isna().any().any():
    counts = image_meta_raw[required_nonmissing].isna().sum()
    raise ValueError(f"Required metadata contain missing values: {counts[counts > 0].to_dict()}")
# Resolve duplicate requests deterministically.
image_meta = image_meta_raw.loc[
    image_meta_raw["download_status_normalised"].isin(VALID_DOWNLOAD_STATUSES)
].copy()
image_meta = image_meta.loc[
    image_meta["request_id"].notna() & image_meta["request_id"].ne("") &
    image_meta["image_path"].notna() & image_meta["image_path"].ne("")
]
image_meta = (
    image_meta.sort_values("_source_row")
    .drop_duplicates("request_id", keep="last")
    .reset_index(drop=True)
)

prepare_local_image_cache(image_meta["image_path"].tolist())
image_meta["resolved_image_path"] = image_meta["image_path"].map(resolve_image_path).map(str)

file_audit = image_meta["resolved_image_path"].apply(inspect_image_file)
image_meta = pd.concat([image_meta, file_audit], axis=1)
image_meta["preflight_valid"] = (
    image_meta["file_exists"] &
    image_meta["file_size_bytes"].ge(CONFIG.min_file_size_bytes)
)
preflight_failed = image_meta.loc[~image_meta["preflight_valid"]].copy()
image_meta = image_meta.loc[image_meta["preflight_valid"]].reset_index(drop=True)

if CONFIG.test_mode:
    image_meta = (
        image_meta.sample(min(CONFIG.test_n_images, len(image_meta)), random_state=RANDOM_STATE)
        .sort_values("request_id")
        .reset_index(drop=True)
    )
    print(f"TEST MODE: processing {len(image_meta):,} images.")
else:
    print(f"PRODUCTION MODE: processing {len(image_meta):,} eligible images.")

if image_meta.empty:
    raise RuntimeError("No valid images remain after metadata and file preflight checks.")

print(f"Raw metadata rows: {len(image_meta_raw):,}")
print(f"Eligible unique requests before file checks: {len(image_meta) + len(preflight_failed):,}")
print(f"Preflight-valid unique requests: {len(image_meta):,}")
print(f"Preflight failures: {len(preflight_failed):,}")
print(f"LSOAs represented: {image_meta[LSOA_ID_COL].nunique():,}")
display(pd.crosstab(image_meta["DeprivationClass"], columns="n_images", margins=True))


PRODUCTION MODE: processing 19,372 eligible images.
Raw metadata rows: 19,372
Eligible unique requests before file checks: 19,372
Preflight-valid unique requests: 19,372
Preflight failures: 0
LSOAs represented: 488


col_0               n_images    All
DeprivationClass                   
High Deprivation        7028   7028
Low Deprivation         6612   6612
Medium Deprivation      5732   5732
All                    19372  19372

In [8]:

PREFLIGHT_FAILURES_CSV = OUTPUT_DIR / "segmentation_preflight_failures.csv"
preflight_failed.to_csv(PREFLIGHT_FAILURES_CSV, index=False)
print(f"Saved preflight-failure audit: {PREFLIGHT_FAILURES_CSV}")


Saved preflight-failure audit: <PROJECT_DIR>/outputs_10points_metadata_once/segmentation_preflight_failures.csv


## 4. Model, label mapping, dataset, and batch collation

In [9]:
def normalise_label(label: Any) -> str:
    return str(label).strip().lower().replace(" ", "_").replace("-", "_")

def load_segformer():
    processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
    model = AutoModelForSemanticSegmentation.from_pretrained(MODEL_NAME)

    device = torch.device("cuda")
    model = model.to(device).eval()

    id2label = {
        int(k): normalise_label(v)
        for k, v in model.config.id2label.items()
    }
    class_map = {
        model_id: label
        for model_id, label in id2label.items()
        if label in CITYSCAPES_19
    }

    missing_labels = sorted(set(CITYSCAPES_19).difference(class_map.values()))
    unexpected_labels = sorted(set(id2label.values()).difference(CITYSCAPES_19))
    if missing_labels:
        raise ValueError(f"Model label mapping is missing classes: {missing_labels}")

    print(f"Model: {MODEL_NAME}")
    print(f"Device: {device}")
    print(f"Mapped classes: {len(class_map)}")
    if unexpected_labels:
        print(f"Model labels not used: {unexpected_labels}")

    return processor, model, device, class_map

class StreetViewDataset(Dataset):
    def __init__(self, frame: pd.DataFrame):
        self.frame = frame.reset_index(drop=True)

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, index: int) -> Dict[str, Any]:
        row = self.frame.iloc[index].to_dict()
        image_path = Path(row["resolved_image_path"])

        try:
            if not image_path.is_file():
                raise FileNotFoundError(str(image_path))
            if image_path.stat().st_size < CONFIG.min_file_size_bytes:
                raise ValueError(
                    f"File smaller than {CONFIG.min_file_size_bytes} bytes"
                )

            # Decode now so corrupt files fail during preflight.
            with Image.open(image_path) as source:
                image = source.convert("RGB")
                image.load()

            if image.width < 32 or image.height < 32:
                raise ValueError(f"Unexpectedly small image dimensions: {image.size}")

            return {
                "ok": True,
                "image": image,
                "original_size": (image.height, image.width),
                "row": row,
                "error": None,
            }
        except Exception as exc:
            return {
                "ok": False,
                "image": None,
                "original_size": None,
                "row": row,
                "error": f"{type(exc).__name__}: {exc}",
            }

def collate_streetview(batch: List[Dict[str, Any]]) -> Dict[str, Any]:
    valid_items = [item for item in batch if item["ok"]]
    invalid_items = [item for item in batch if not item["ok"]]
    return {"valid": valid_items, "invalid": invalid_items}

## 5. Inference helpers, QC metrics, checkpointing, and retries

In [10]:
SEG_COLS = [f"seg_{name}" for name in CITYSCAPES_19]

def empty_feature_record(row: Dict[str, Any], status: str, error: Optional[str]) -> Dict[str, Any]:
    out = dict(row)
    for col in SEG_COLS:
        out[col] = np.nan
    out.update({
        "segmentation_status": status,
        "segmentation_error": error,
        "segmentation_attempted_at_utc": datetime.now(timezone.utc).isoformat(),
        "segmentation_model": MODEL_NAME,
        "segmentation_proportion_sum": np.nan,
        "segmentation_entropy": np.nan,
        "dominant_segmentation_class": None,
        "dominant_segmentation_proportion": np.nan,
        "low_information_flag": None,
        "segmentation_qc": "fail",
    })
    return out

def prediction_to_features(
    prediction: np.ndarray,
    row: Dict[str, Any],
    class_map: Dict[int, str]
) -> Dict[str, Any]:
    total_pixels = int(prediction.size)
    if total_pixels <= 0:
        raise ValueError("Empty prediction mask")

    proportions = {name: 0.0 for name in CITYSCAPES_19}
    unique_ids, counts = np.unique(prediction, return_counts=True)

    for model_id, count in zip(unique_ids.tolist(), counts.tolist()):
        class_name = class_map.get(int(model_id))
        if class_name is not None:
            proportions[class_name] += float(count) / total_pixels

    values = np.array([proportions[name] for name in CITYSCAPES_19], dtype=float)
    proportion_sum = float(values.sum())

    # Normalised entropy: higher values mean a more varied scene.
    positive = values[values > 0]
    entropy = float(-(positive * np.log(positive)).sum() / np.log(len(CITYSCAPES_19)))

    dominant_index = int(values.argmax())
    dominant_class = CITYSCAPES_19[dominant_index]
    dominant_proportion = float(values[dominant_index])

    qc_ok = abs(proportion_sum - 1.0) <= CONFIG.qc_sum_tolerance

    out = dict(row)
    for name, value in proportions.items():
        out[f"seg_{name}"] = value

    out.update({
        "segmentation_status": "ok" if qc_ok else "failed_qc",
        "segmentation_error": None if qc_ok else (
            f"Class proportions sum to {proportion_sum:.8f}, expected approximately 1.0"
        ),
        "segmentation_attempted_at_utc": datetime.now(timezone.utc).isoformat(),
        "segmentation_model": MODEL_NAME,
        "segmentation_proportion_sum": proportion_sum,
        "segmentation_entropy": entropy,
        "dominant_segmentation_class": dominant_class,
        "dominant_segmentation_proportion": dominant_proportion,
        # Flag dominant scenes for review; they may still be valid.
        "low_information_flag": dominant_proportion >= CONFIG.low_information_threshold,
        "segmentation_qc": "pass" if qc_ok else "fail",
    })
    return out

def autocast_context(device: torch.device):
    if device.type != "cuda" or not CONFIG.use_amp:
        return nullcontext()

    dtype = torch.float16 if CONFIG.amp_dtype == "float16" else torch.bfloat16
    return torch.autocast(device_type="cuda", dtype=dtype)

def infer_valid_items(
    valid_items: List[Dict[str, Any]],
    processor,
    model,
    device: torch.device,
    class_map: Dict[int, str],
) -> List[Dict[str, Any]]:
    if not valid_items:
        return []

    images = [item["image"] for item in valid_items]
    inputs = processor(images=images, return_tensors="pt")
    inputs = {key: value.to(device, non_blocking=True) for key, value in inputs.items()}

    with torch.inference_mode(), autocast_context(device):
        logits = model(**inputs).logits

    results = []
    for batch_index, item in enumerate(valid_items):
        original_height, original_width = item["original_size"]
        resized_logits = F.interpolate(
            logits[batch_index:batch_index + 1],
            size=(original_height, original_width),
            mode="bilinear",
            align_corners=False,
        )
        prediction = (
            resized_logits.argmax(dim=1)
            .squeeze(0)
            .detach()
            .cpu()
            .numpy()
            .astype(np.int16)
        )
        results.append(prediction_to_features(prediction, item["row"], class_map))

    del inputs, logits
    return results

def infer_batch_with_fallback(
    valid_items: List[Dict[str, Any]],
    processor,
    model,
    device: torch.device,
    class_map: Dict[int, str],
) -> List[Dict[str, Any]]:
    if not valid_items:
        return []

    try:
        return infer_valid_items(valid_items, processor, model, device, class_map)
    except RuntimeError as exc:
        # Isolate a bad image or memory failure by retrying items separately.
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

        results = []
        for item in valid_items:
            success = False
            last_error = None
            for attempt in range(1, CONFIG.max_retry_attempts + 1):
                try:
                    results.extend(
                        infer_valid_items([item], processor, model, device, class_map)
                    )
                    success = True
                    break
                except Exception as single_exc:
                    last_error = single_exc
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
                    gc.collect()
                    time.sleep(min(attempt, 2))
            if not success:
                results.append(
                    empty_feature_record(
                        item["row"],
                        status="failed",
                        error=f"{type(last_error).__name__}: {last_error}",
                    )
                )
        return results

def consolidate_results(new_rows: List[Dict[str, Any]]) -> pd.DataFrame:
    frames = []

    if SEG_IMAGE_FEATURES_CSV.exists():
        previous = pd.read_csv(SEG_IMAGE_FEATURES_CSV, low_memory=False)
        previous["request_id"] = previous["request_id"].astype(str)
        frames.append(previous)

    if new_rows:
        current = pd.DataFrame(new_rows)
        current["request_id"] = current["request_id"].astype(str)
        frames.append(current)

    if not frames:
        return pd.DataFrame()

    # Later checkpoints replace earlier failures for the same request.
    combined = (
        pd.concat(frames, ignore_index=True, sort=False)
        .drop_duplicates("request_id", keep="last")
        .sort_values("request_id")
        .reset_index(drop=True)
    )
    return combined

def save_checkpoint(new_rows: List[Dict[str, Any]], batch_number: int) -> None:
    if not new_rows:
        return

    checkpoint_path = CHECKPOINT_DIR / f"segmentation_part_{batch_number:06d}.parquet"
    pd.DataFrame(new_rows).to_parquet(checkpoint_path, index=False)

    combined = consolidate_results(new_rows)
    combined.to_csv(SEG_IMAGE_FEATURES_CSV, index=False)

def load_successful_request_ids() -> set:
    if CONFIG.force_reprocess_successes or not SEG_IMAGE_FEATURES_CSV.exists():
        return set()

    existing = pd.read_csv(
        SEG_IMAGE_FEATURES_CSV,
        usecols=lambda col: col in {"request_id", "segmentation_status"},
        low_memory=False,
    )
    successful = existing.loc[
        existing["segmentation_status"].eq("ok"), "request_id"
    ].astype(str)
    return set(successful)

## 6. Run resumable segmentation

In [11]:
def run_segmentation(image_meta: pd.DataFrame) -> pd.DataFrame:
    successful_ids = load_successful_request_ids()

    todo = image_meta.loc[
        ~image_meta["request_id"].astype(str).isin(successful_ids)
    ].copy().reset_index(drop=True)

    print(f"Already completed successfully: {len(successful_ids):,}")
    print(f"Images to process or retry: {len(todo):,}")

    if todo.empty:
        print("No images remain. Loading existing feature file.")
        return pd.read_csv(SEG_IMAGE_FEATURES_CSV, low_memory=False)

    processor, model, device, class_map = load_segformer()

    dataset = StreetViewDataset(todo)
    loader = DataLoader(
        dataset,
        batch_size=CONFIG.batch_size,
        shuffle=False,
        num_workers=CONFIG.num_workers,
        pin_memory=True,
        persistent_workers=CONFIG.num_workers > 0,
        collate_fn=collate_streetview,
    )

    pending_rows: List[Dict[str, Any]] = []
    all_new_rows: List[Dict[str, Any]] = []
    start_time = time.time()

    progress = tqdm(loader, total=len(loader), desc="SegFormer inference", unit="batch")

    for batch_number, batch in enumerate(progress, start=1):
        # Keep decode failures returned by the Dataset.
        for invalid in batch["invalid"]:
            pending_rows.append(
                empty_feature_record(
                    invalid["row"],
                    status="failed",
                    error=invalid["error"],
                )
            )

        # Fall back to single-image inference when a batch fails.
        inferred = infer_batch_with_fallback(
            batch["valid"], processor, model, device, class_map
        )
        pending_rows.extend(inferred)

        if (
            batch_number % CONFIG.checkpoint_every_batches == 0
            or batch_number == len(loader)
        ):
            save_checkpoint(pending_rows, batch_number)
            all_new_rows.extend(pending_rows)
            pending_rows = []

        processed = min(batch_number * CONFIG.batch_size, len(todo))
        elapsed = max(time.time() - start_time, 1e-6)
        rate = processed / elapsed
        progress.set_postfix({
            "images": f"{processed:,}/{len(todo):,}",
            "img_s": f"{rate:.2f}",
        })

    final = consolidate_results(all_new_rows)
    final.to_csv(SEG_IMAGE_FEATURES_CSV, index=False)

    failures = final.loc[
        ~final["segmentation_status"].eq("ok")
    ].copy()
    failures.to_csv(SEG_FAILURES_CSV, index=False)

    summary = {
        **runtime_report,
        "model_name": MODEL_NAME,
        "configuration": asdict(CONFIG),
        "metadata_valid_images": int(len(image_meta)),
        "records_in_feature_file": int(len(final)),
        "successful_segmentations": int(final["segmentation_status"].eq("ok").sum()),
        "failed_or_failed_qc": int((~final["segmentation_status"].eq("ok")).sum()),
        "elapsed_seconds_this_run": round(time.time() - start_time, 2),
    }
    with open(SEG_RUN_SUMMARY_JSON, "w") as f:
        json.dump(summary, f, indent=2)

    return final

seg_img = run_segmentation(image_meta)

print("\nSegmentation status:")
display(seg_img["segmentation_status"].value_counts(dropna=False).to_frame("n_images"))

print("\nOutput file:", SEG_IMAGE_FEATURES_CSV)
print("Failure file:", SEG_FAILURES_CSV)
print("Run summary:", SEG_RUN_SUMMARY_JSON)

Already completed successfully: 0
Images to process or retry: 19,372


preprocessor_config.json:   0%|          | 0.00/272 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.68k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 15.0MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

...

SegFormer inference:   0%|          | 0/4843 [00:00<?, ?batch/s]

model.safetensors: downloading bytes:           |  0.00B            


Segmentation status:


                     n_images
segmentation_status          
ok                      19372


Output file: <PROJECT_DIR>/outputs_10points_metadata_once/segmentation_image_features_19class.csv
Failure file: <PROJECT_DIR>/outputs_10points_metadata_once/segmentation_failures.csv
Run summary: <PROJECT_DIR>/outputs_10points_metadata_once/segmentation_run_summary.json


## 7. Image-level quality-control audit

In [12]:
successful = seg_img.loc[seg_img["segmentation_status"].eq("ok")].copy()

if successful.empty:
    raise RuntimeError("No successful segmentations are available for quality control.")

qc_summary = pd.Series({
    "successful_images": len(successful),
    "failed_images": int((~seg_img["segmentation_status"].eq("ok")).sum()),
    "proportion_sum_min": successful["segmentation_proportion_sum"].min(),
    "proportion_sum_max": successful["segmentation_proportion_sum"].max(),
    "proportion_sum_mean": successful["segmentation_proportion_sum"].mean(),
    "low_information_flagged": successful["low_information_flag"].fillna(False).sum(),
    "unique_lsoas": successful[LSOA_ID_COL].nunique(),
})
display(qc_summary.to_frame("value"))

print("\nClass-proportion distributions:")
display(successful[SEG_COLS].describe().T)

print("\nDominant classes:")
display(
    successful["dominant_segmentation_class"]
    .value_counts(dropna=False)
    .rename("n_images")
    .to_frame()
)

# Successful rows should close to one.
bad_sums = successful.loc[
    ~successful["segmentation_proportion_sum"].between(
        1.0 - CONFIG.qc_sum_tolerance,
        1.0 + CONFIG.qc_sum_tolerance,
    )
]
assert bad_sums.empty, (
    f"{len(bad_sums)} successful rows failed the class-proportion sum check."
)

                           value
successful_images        19372.0
failed_images                0.0
proportion_sum_min           1.0
proportion_sum_max           1.0
proportion_sum_mean          1.0
low_information_flagged     13.0
unique_lsoas               488.0


Class-proportion distributions:


                     count      mean       std  min       25%       50%  \
seg_road           19372.0  0.296084  0.097847  0.0  0.243725  0.318878   
seg_sidewalk       19372.0  0.041100  0.047340  0.0  0.006685  0.025527   
seg_building       19372.0  0.106793  0.102350  0.0  0.034265  0.081165   
seg_wall           19372.0  0.012465  0.026894  0.0  0.000000  0.002122   
seg_fence          19372.0  0.015080  0.027791  0.0  0.000005  0.003164   
seg_pole           19372.0  0.003153  0.005081  0.0  0.000205  0.001288   
seg_traffic_light  19372.0  0.000094  0.000679  0.0  0.000000  0.000000   
seg_traffic_sign   19372.0  0.000598  0.002048  0.0  0.000000  0.000017   
seg_vegetation     19372.0  0.140928  0.144049  0.0  0.040447  0.091851   
seg_terrain        19372.0  0.036734  0.057769  0.0  0.001791  0.013329   
seg_sky            19372.0  0.323674  0.105989  0.0  0.275020  0.353978   
seg_person         19372.0  0.000736  0.002935  0.0  0.000000  0.000012   
seg_rider          19372.


Dominant classes:


                             n_images
dominant_segmentation_class          
sky                             10742
road                             5233
vegetation                       2299
building                          992
terrain                            64
car                                16
wall                               15
sidewalk                            5
bus                                 4
fence                               2

## 8. Optional visual QC: original images and predicted masks

This cell re-runs inference for a small, reproducible sample and saves overlays for human inspection. It is a quality-control step, not part of model training.

Review whether buildings, roads, vegetation, sky, pavement, vehicles, and people are generally represented plausibly. Do not expect every pixel to be correct.

In [13]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

# Cityscapes palette, ordered to match CITYSCAPES_19.
CITYSCAPES_PALETTE = [
    (128, 64, 128), (244, 35, 232), (70, 70, 70), (102, 102, 156),
    (190, 153, 153), (153, 153, 153), (250, 170, 30), (220, 220, 0),
    (107, 142, 35), (152, 251, 152), (70, 130, 180), (220, 20, 60),
    (255, 0, 0), (0, 0, 142), (0, 0, 70), (0, 60, 100),
    (0, 80, 100), (0, 0, 230), (119, 11, 32)
]
CITYSCAPES_CMAP = ListedColormap(
    np.array(CITYSCAPES_PALETTE, dtype=float) / 255.0
)

def save_visual_qc(
    metadata: pd.DataFrame,
    n_images: int = 24,
    random_state: int = 42,
) -> None:
    sample = metadata.sample(
        n=min(n_images, len(metadata)),
        random_state=random_state
    ).reset_index(drop=True)

    processor, model, device, class_map = load_segformer()
    model_id_to_city_index = {
        model_id: CITYSCAPES_19.index(class_name)
        for model_id, class_name in class_map.items()
    }

    for _, row in tqdm(sample.iterrows(), total=len(sample), desc="Saving QC masks"):
        image_path = Path(row["resolved_image_path"])
        with Image.open(image_path) as source:
            image = source.convert("RGB")
            image.load()

        inputs = processor(images=image, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.inference_mode(), autocast_context(device):
            logits = model(**inputs).logits

        logits = F.interpolate(
            logits,
            size=(image.height, image.width),
            mode="bilinear",
            align_corners=False,
        )
        model_prediction = logits.argmax(dim=1).squeeze(0).cpu().numpy()

        city_prediction = np.zeros_like(model_prediction, dtype=np.int16)
        for model_id, city_index in model_id_to_city_index.items():
            city_prediction[model_prediction == model_id] = city_index

        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        axes[0].imshow(image)
        axes[0].set_title("Original")
        axes[1].imshow(city_prediction, cmap=CITYSCAPES_CMAP, vmin=0, vmax=18)
        axes[1].set_title("Segmentation mask")
        axes[2].imshow(image)
        axes[2].imshow(city_prediction, cmap=CITYSCAPES_CMAP, vmin=0, vmax=18, alpha=0.45)
        axes[2].set_title("Overlay")

        for axis in axes:
            axis.axis("off")

        fig.suptitle(
            f"{row['request_id']} | {row[LSOA_ID_COL]} | heading={row.get('heading_angle')}"
        )
        fig.tight_layout()

        output_path = QC_DIR / f"qc_{row['request_id']}.png"
        fig.savefig(output_path, dpi=160, bbox_inches="tight")
        plt.close(fig)

    print(f"Saved visual QC files to: {QC_DIR}")

if CONFIG.save_qc_masks:
    save_visual_qc(
        image_meta,
        n_images=CONFIG.n_qc_masks,
        random_state=RANDOM_STATE
    )

Loading weights:   0%|          | 0/208 [00:00<?, ?it/s]

Model: nvidia/segformer-b0-finetuned-cityscapes-1024-1024
Device: cuda
Mapped classes: 19


Saving QC masks:   0%|          | 0/24 [00:00<?, ?it/s]

Saved visual QC files to: <PROJECT_DIR>/outputs_10points_metadata_once/segmentation_qc



## 9. Aggregate to LSOA level and construct final Spatial+-inspired folds

The 19 semantic proportions are compositional and sum to one. The output therefore includes interpretable raw proportions, grouped streetscape indicators, and centred log-ratio features. Final folds are constructed only after aggregation, using complete-linkage spatial blocks and a target/coverage/feature-aware block assignment optimiser.


In [14]:

from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import StandardScaler

successful = seg_img.loc[seg_img["segmentation_status"].eq("ok")].copy()
BASE_TARGET_COLUMNS = [LSOA_ID_COL, "IMDRank", "IMDDecile", "DeprivationClass", "ClassIdx"]
missing = [c for c in BASE_TARGET_COLUMNS if c not in successful.columns]
if missing:
    raise KeyError(f"Missing target columns: {missing}")

# Ignore fold columns left by earlier runs.
successful = successful.drop(columns=["spatial_fold", "spatial_block"], errors="ignore")

consistency = successful.groupby(LSOA_ID_COL)[BASE_TARGET_COLUMNS[1:]].nunique(dropna=False)
conflicts = consistency.loc[(consistency > 1).any(axis=1)]
if not conflicts.empty:
    raise ValueError(f"{len(conflicts)} LSOAs contain conflicting deprivation metadata.")

grouped = successful.groupby(LSOA_ID_COL, sort=True)
means = grouped[SEG_COLS].mean().add_suffix("_mean")
stds = grouped[SEG_COLS].std().add_suffix("_std")
medians = grouped[SEG_COLS].median().add_suffix("_median")
coverage = grouped.agg(
    n_images_segmented=("request_id", "size"),
    n_sampling_points=("point_id", "nunique"),
    n_headings=("heading_angle", "nunique"),
    lsoa_lat=("sample_lat", "median"),
    lsoa_lon=("sample_lon", "median"),
    mean_segmentation_entropy=("segmentation_entropy", "mean"),
    sd_segmentation_entropy=("segmentation_entropy", "std"),
    n_low_information_images=("low_information_flag", "sum"),
)
if "year" in successful.columns:
    coverage = coverage.join(grouped.year.agg(
        median_image_year="median", min_image_year="min",
        max_image_year="max", n_unique_image_years="nunique"
    ))

meta = successful[BASE_TARGET_COLUMNS].drop_duplicates(LSOA_ID_COL).set_index(LSOA_ID_COL)
lsoa_features = meta.join(means).join(stds).join(medians).join(coverage).reset_index()

# Theory-led streetscape proxies, not causal measures.
def sum_mean(names: Sequence[str]) -> pd.Series:
    return lsoa_features[[f"seg_{n}_mean" for n in names]].sum(axis=1)

lsoa_features["green_natural_mean"] = sum_mean(["vegetation", "terrain"])
lsoa_features["built_form_mean"] = sum_mean(["building", "wall", "fence"])
lsoa_features["pedestrian_realm_mean"] = sum_mean(["sidewalk"])
lsoa_features["road_transport_mean"] = sum_mean([
    "road", "car", "truck", "bus", "train", "motorcycle", "bicycle"
])
lsoa_features["street_furniture_mean"] = sum_mean(["pole", "traffic_light", "traffic_sign"])
lsoa_features["active_travel_people_mean"] = sum_mean(["person", "rider", "bicycle"])
lsoa_features["sky_openness_mean"] = lsoa_features["seg_sky_mean"]

# CLR transform of the mean composition.
P = lsoa_features[[f"seg_{n}_mean" for n in CITYSCAPES_19]].to_numpy(float)
if not np.isfinite(P).all() or (P < 0).any():
    raise ValueError("Invalid semantic proportions found before CLR transformation.")
P = np.clip(P, CONFIG.clr_pseudocount, None)
P = P / P.sum(axis=1, keepdims=True)
clr = np.log(P) - np.log(P).mean(axis=1, keepdims=True)
for j, name in enumerate(CITYSCAPES_19):
    lsoa_features[f"clr_{name}"] = clr[:, j]

lsoa_features["coverage_status"] = np.select(
    [
        lsoa_features.n_images_segmented < CONFIG.min_images_per_lsoa,
        lsoa_features.n_images_segmented < CONFIG.preferred_images_per_lsoa,
    ],
    ["below_minimum", "acceptable_but_below_preferred"],
    default="preferred_coverage",
)
lsoa_features["low_information_fraction"] = (
    lsoa_features.n_low_information_images / lsoa_features.n_images_segmented
)

# Stage 1: complete-linkage spatial micro-blocks
def project_lon_lat_km(df: pd.DataFrame) -> np.ndarray:
    mean_lat = np.deg2rad(df["lsoa_lat"].mean())
    x = 111.320 * np.cos(mean_lat) * df["lsoa_lon"].to_numpy(float)
    y = 110.574 * df["lsoa_lat"].to_numpy(float)
    return np.column_stack([x, y])

coords = project_lon_lat_km(lsoa_features)
lsoa_features["x_km"] = coords[:, 0]
lsoa_features["y_km"] = coords[:, 1]

n_blocks = min(
    max(CONFIG.n_spatial_blocks, CONFIG.n_spatial_folds * 10),
    len(lsoa_features),
)
if n_blocks < CONFIG.n_spatial_folds:
    raise ValueError("Too few spatial blocks for the requested number of folds.")

block_model = AgglomerativeClustering(
    n_clusters=n_blocks,
    linkage=CONFIG.spatial_block_linkage,
    metric="euclidean",
)
lsoa_features["spatial_block"] = block_model.fit_predict(coords).astype(int)

# Stage 2: assign complete blocks using location, target, coverage and semantics
classes = sorted(lsoa_features["DeprivationClass"].astype(str).unique())
if len(classes) != 3:
    raise ValueError(f"Expected three deprivation classes; found {classes}")

# Use a compact fold-design set so correlated semantic columns do not dominate.
FOLD_FEATURE_COLUMNS = [
    "green_natural_mean", "built_form_mean", "pedestrian_realm_mean",
    "road_transport_mean", "street_furniture_mean",
    "active_travel_people_mean", "sky_openness_mean",
    "mean_segmentation_entropy", "low_information_fraction",
]

class_counts = (
    pd.crosstab(lsoa_features["spatial_block"], lsoa_features["DeprivationClass"])
    .reindex(columns=classes, fill_value=0)
)
block_base = lsoa_features.groupby("spatial_block", as_index=False).agg(
    n_lsoas=(LSOA_ID_COL, "size"),
    n_images=("n_images_segmented", "sum"),
    centroid_x_km=("x_km", "mean"),
    centroid_y_km=("y_km", "mean"),
)
block_features = lsoa_features.groupby("spatial_block")[FOLD_FEATURE_COLUMNS].mean()
blocks = (
    block_base
    .merge(class_counts.reset_index(), on="spatial_block", validate="one_to_one")
    .merge(block_features.reset_index(), on="spatial_block", validate="one_to_one")
    .sort_values("spatial_block")
    .reset_index(drop=True)
)

# Scale only the fold-design summaries, not the modelling matrix.
feature_matrix = StandardScaler().fit_transform(blocks[FOLD_FEATURE_COLUMNS])
size_class_columns = ["n_lsoas", "n_images", *classes]
size_class_matrix = blocks[size_class_columns].to_numpy(float)
centroids = blocks[["centroid_x_km", "centroid_y_km"]].to_numpy(float)
block_ids = blocks["spatial_block"].to_numpy(int)


def fold_objective(assignment: np.ndarray) -> float:
    k = CONFIG.n_spatial_folds
    if any(np.sum(assignment == f) == 0 for f in range(k)):
        return float("inf")

    fold_totals = np.vstack([
        size_class_matrix[assignment == f].sum(axis=0) for f in range(k)
    ])
    targets = size_class_matrix.sum(axis=0) / k
    rel = (fold_totals - targets) / np.maximum(targets, 1.0)

    score = CONFIG.fold_weight_lsoas * np.mean(rel[:, 0] ** 2)
    score += CONFIG.fold_weight_images * np.mean(rel[:, 1] ** 2)
    score += CONFIG.fold_weight_classes * np.mean(rel[:, 2:] ** 2)

    # Feature balance: weighted fold means should remain representative.
    global_feature_mean = np.average(
        feature_matrix, axis=0, weights=np.maximum(size_class_matrix[:, 0], 1)
    )
    feature_penalty = 0.0
    for f in range(k):
        mask = assignment == f
        fold_mean = np.average(
            feature_matrix[mask], axis=0,
            weights=np.maximum(size_class_matrix[mask, 0], 1),
        )
        feature_penalty += np.mean((fold_mean - global_feature_mean) ** 2)
    score += CONFIG.fold_weight_features * feature_penalty / k

    # Geographic coherence: penalise within-fold dispersion of block centroids.
    compactness = 0.0
    for f in range(k):
        mask = assignment == f
        weights = np.maximum(size_class_matrix[mask, 0], 1)
        centre = np.average(centroids[mask], axis=0, weights=weights)
        sq_dist = ((centroids[mask] - centre) ** 2).sum(axis=1)
        compactness += np.average(sq_dist, weights=weights)
    spatial_scale = max(np.var(centroids, axis=0).sum(), 1e-9)
    score += CONFIG.fold_weight_compactness * (compactness / k) / spatial_scale
    return float(score)


def optimise_blocks() -> tuple[np.ndarray, float]:
    master = np.random.default_rng(RANDOM_STATE)
    k = CONFIG.n_spatial_folds
    best_assignment = None
    best_score = float("inf")
    target = size_class_matrix.sum(axis=0) / k

    for restart in range(CONFIG.fold_optimizer_restarts):
        rng = np.random.default_rng(int(master.integers(0, 2**32 - 1)))
        priority = (
            size_class_matrix[:, 0] / max(target[0], 1) +
            size_class_matrix[:, 1] / max(target[1], 1) +
            size_class_matrix[:, 2:].max(axis=1) /
            max(target[2:].max(), 1) +
            rng.normal(0, 0.05, len(blocks))
        )
        order = np.argsort(-priority)
        assignment = np.full(len(blocks), -1, dtype=int)

        # Seed with geographically separated large blocks.
        seeds = [int(order[0])]
        while len(seeds) < k:
            candidates = [i for i in order if i not in seeds]
            d = [min(np.linalg.norm(centroids[i] - centroids[j]) for j in seeds)
                 for i in candidates]
            seeds.append(int(candidates[int(np.argmax(d))]))
        for f, i in enumerate(seeds):
            assignment[i] = f

        # Greedy construction using a partial fold-balance proxy.
        fold_totals = np.zeros((k, size_class_matrix.shape[1]), dtype=float)
        fold_feature_sums = np.zeros((k, feature_matrix.shape[1]), dtype=float)
        fold_feature_weights = np.zeros(k, dtype=float)
        for f, i in enumerate(seeds):
            fold_totals[f] += size_class_matrix[i]
            w = max(size_class_matrix[i, 0], 1)
            fold_feature_sums[f] += feature_matrix[i] * w
            fold_feature_weights[f] += w

        for i in [j for j in order if assignment[j] < 0]:
            candidate_scores = []
            for f in range(k):
                trial_totals = fold_totals.copy()
                trial_totals[f] += size_class_matrix[i]
                rel = (trial_totals - target) / np.maximum(target, 1.0)
                s = (
                    CONFIG.fold_weight_lsoas * np.mean(rel[:, 0] ** 2) +
                    CONFIG.fold_weight_images * np.mean(rel[:, 1] ** 2) +
                    CONFIG.fold_weight_classes * np.mean(rel[:, 2:] ** 2)
                )
                # Slightly prefer nearby blocks during construction.
                assigned_centres = centroids[assignment == f]
                if len(assigned_centres):
                    s += 0.02 * np.min(np.linalg.norm(assigned_centres - centroids[i], axis=1))
                candidate_scores.append(s)
            chosen = int(np.argmin(candidate_scores))
            assignment[i] = chosen
            fold_totals[chosen] += size_class_matrix[i]
            w = max(size_class_matrix[i, 0], 1)
            fold_feature_sums[chosen] += feature_matrix[i] * w
            fold_feature_weights[chosen] += w

        current = fold_objective(assignment)
        temperature = 0.08
        for _ in range(CONFIG.fold_optimizer_iterations):
            proposal = assignment.copy()
            if rng.random() < 0.65:
                i = int(rng.integers(len(proposal)))
                old = proposal[i]
                if np.sum(proposal == old) <= 1:
                    continue
                proposal[i] = int(rng.choice([f for f in range(k) if f != old]))
            else:
                i, j = rng.choice(len(proposal), 2, replace=False)
                if proposal[i] == proposal[j]:
                    continue
                proposal[i], proposal[j] = proposal[j], proposal[i]

            proposed = fold_objective(proposal)
            delta = proposed - current
            if delta < 0 or rng.random() < np.exp(-delta / max(temperature, 1e-9)):
                assignment, current = proposal, proposed
            temperature *= 0.99965

        if current < best_score:
            best_assignment, best_score = assignment.copy(), current

    if best_assignment is None:
        raise RuntimeError("Final spatial fold optimisation failed.")
    return best_assignment, float(best_score)


assignment, fold_score = optimise_blocks()
block_to_fold = {int(block_ids[i]): int(assignment[i] + 1) for i in range(len(block_ids))}
lsoa_features["spatial_fold"] = lsoa_features["spatial_block"].map(block_to_fold).astype(int)

# Fold diagnostics and validation gates
fold_map_columns = [
    LSOA_ID_COL, "lsoa_lat", "lsoa_lon", "x_km", "y_km",
    "DeprivationClass", "ClassIdx", "n_images_segmented",
    "spatial_block", "spatial_fold",
]
lsoa_fold_map = lsoa_features[fold_map_columns].copy()

image_class_counts = pd.crosstab(
    lsoa_features["spatial_fold"], lsoa_features["DeprivationClass"],
    values=lsoa_features["n_images_segmented"], aggfunc="sum"
).fillna(0).add_prefix("images_")
lsoa_class_counts = pd.crosstab(
    lsoa_features["spatial_fold"], lsoa_features["DeprivationClass"]
).add_prefix("lsoas_")
spatial_fold_audit = (
    lsoa_features.groupby("spatial_fold")
    .agg(
        n_lsoas=(LSOA_ID_COL, "size"),
        n_images=("n_images_segmented", "sum"),
        n_points=("n_sampling_points", "sum"),
        n_blocks=("spatial_block", "nunique"),
    )
    .join(image_class_counts)
    .join(lsoa_class_counts)
    .reset_index()
)

expected_folds = list(range(1, CONFIG.n_spatial_folds + 1))
if sorted(lsoa_features["spatial_fold"].unique().tolist()) != expected_folds:
    raise AssertionError("Not all requested folds were produced.")
if lsoa_features.groupby(LSOA_ID_COL)["spatial_fold"].nunique().max() != 1:
    raise AssertionError("LSOA leakage detected.")
if lsoa_features.groupby("spatial_block")["spatial_fold"].nunique().max() != 1:
    raise AssertionError("A spatial block was split across folds.")

lsoa_class_table = pd.crosstab(lsoa_features["spatial_fold"], lsoa_features["DeprivationClass"])
if (lsoa_class_table < CONFIG.minimum_fold_class_count).any().any():
    raise ValueError(
        "At least one fold has fewer than "
        f"{CONFIG.minimum_fold_class_count} LSOAs in a deprivation class."
    )

lsoa_cv = spatial_fold_audit["n_lsoas"].std(ddof=1) / spatial_fold_audit["n_lsoas"].mean()
image_cv = spatial_fold_audit["n_images"].std(ddof=1) / spatial_fold_audit["n_images"].mean()
if lsoa_cv > CONFIG.maximum_fold_size_cv or image_cv > CONFIG.maximum_fold_size_cv:
    raise ValueError(
        f"Fold-size imbalance exceeds threshold: LSOA CV={lsoa_cv:.3f}, "
        f"image CV={image_cv:.3f}, maximum={CONFIG.maximum_fold_size_cv:.3f}."
    )

# Standardised mean differences between each fold and the full sample.
feature_values = lsoa_features[FOLD_FEATURE_COLUMNS].replace([np.inf, -np.inf], np.nan)
if feature_values.isna().any().any():
    raise ValueError("Missing/non-finite fold-design features found.")
feature_sd = feature_values.std(ddof=0).replace(0, 1.0)
feature_global = feature_values.mean()
feature_smd_rows = []
for fold in expected_folds:
    fold_mean = lsoa_features.loc[
        lsoa_features["spatial_fold"].eq(fold), FOLD_FEATURE_COLUMNS
    ].mean()
    smd = ((fold_mean - feature_global) / feature_sd).abs()
    feature_smd_rows.append({"spatial_fold": fold, **smd.to_dict(), "max_abs_smd": smd.max()})
fold_feature_balance = pd.DataFrame(feature_smd_rows)
if fold_feature_balance["max_abs_smd"].max() > CONFIG.maximum_feature_smd:
    raise ValueError(
        "Semantic feature imbalance exceeds the configured maximum absolute SMD: "
        f"{fold_feature_balance['max_abs_smd'].max():.3f}."
    )

SPATIAL_FOLD_MAP_CSV = OUTPUT_DIR / "lsoa_spatial_plus_fold_mapping.csv"
SPATIAL_FOLD_AUDIT_CSV = OUTPUT_DIR / "spatial_plus_fold_audit.csv"
SPATIAL_FOLD_FEATURE_BALANCE_CSV = OUTPUT_DIR / "spatial_plus_fold_feature_balance.csv"
lsoa_fold_map.to_csv(SPATIAL_FOLD_MAP_CSV, index=False)
spatial_fold_audit.to_csv(SPATIAL_FOLD_AUDIT_CSV, index=False)
fold_feature_balance.to_csv(SPATIAL_FOLD_FEATURE_BALANCE_CSV, index=False)

# Replace stale image-level folds with the final mapping.
seg_img = seg_img.drop(columns=["spatial_fold", "spatial_block"], errors="ignore").merge(
    lsoa_fold_map[[LSOA_ID_COL, "spatial_block", "spatial_fold"]],
    on=LSOA_ID_COL, how="left", validate="many_to_one",
)
seg_img.to_csv(SEG_IMAGE_FEATURES_CSV, index=False)

# Retain low-coverage LSOAs for sensitivity analysis.
lsoa_features.to_csv(SEG_LSOA_FEATURES_CSV, index=False)

print(f"Best final fold objective: {fold_score:.6f}")
print(f"Fold-size CV — LSOAs: {lsoa_cv:.4f}; images: {image_cv:.4f}")
print(f"Maximum fold feature absolute SMD: {fold_feature_balance['max_abs_smd'].max():.4f}")
print(f"Saved LSOA feature table: {SEG_LSOA_FEATURES_CSV}")
print(f"Saved fold mapping: {SPATIAL_FOLD_MAP_CSV}")
print(f"Saved fold audit: {SPATIAL_FOLD_AUDIT_CSV}")
print(f"Saved fold feature-balance audit: {SPATIAL_FOLD_FEATURE_BALANCE_CSV}")

display(spatial_fold_audit)
display(pd.crosstab(lsoa_features.spatial_fold, lsoa_features.DeprivationClass, margins=True))
display(fold_feature_balance)
display(lsoa_features.coverage_status.value_counts(dropna=False).to_frame("n_lsoas"))
display(lsoa_features.head())




Best final fold objective: 0.158868
Fold-size CV — LSOAs: 0.0117; images: 0.0128
Maximum fold feature absolute SMD: 0.2343
Saved LSOA feature table: <PROJECT_DIR>/outputs_10points_metadata_once/segmentation_lsoa_features_19class_mean_sd.csv
Saved fold mapping: <PROJECT_DIR>/outputs_10points_metadata_once/lsoa_spatial_plus_fold_mapping.csv
Saved fold audit: <PROJECT_DIR>/outputs_10points_metadata_once/spatial_plus_fold_audit.csv
Saved fold feature-balance audit: <PROJECT_DIR>/outputs_10points_metadata_once/spatial_plus_fold_feature_balance.csv


   spatial_fold  n_lsoas  n_images  n_points  n_blocks  \
0             1       98      3860       965        25   
1             2       98      3888       972        22   
2             3       99      3948       987        22   
3             4       96      3812       953        27   
4             5       97      3864       966        24   

   images_High Deprivation  images_Low Deprivation  images_Medium Deprivation  \
0                     1356                    1348                       1156   
1                     1396                    1336                       1156   
2                     1480                    1316                       1152   
3                     1396                    1304                       1112   
4                     1400                    1308                       1156   

   lsoas_High Deprivation  lsoas_Low Deprivation  lsoas_Medium Deprivation  
0                      34                     34                        30  
1         

DeprivationClass  High Deprivation  Low Deprivation  Medium Deprivation  All
spatial_fold                                                                
1                               34               34                  30   98
2                               35               34                  29   98
3                               37               33                  29   99
4                               35               33                  28   96
5                               35               33                  29   97
All                            176              167                 145  488

   spatial_fold  green_natural_mean  built_form_mean  pedestrian_realm_mean  \
0             1            0.029373         0.059264               0.090052   
1             2            0.020457         0.038236               0.029192   
2             3            0.166036         0.106730               0.077693   
3             4            0.110525         0.137025               0.007582   
4             5            0.009730         0.071824               0.048682   

   road_transport_mean  street_furniture_mean  active_travel_people_mean  \
0             0.213489               0.006069                   0.115581   
1             0.044843               0.013045                   0.042386   
2             0.062250               0.087953                   0.027091   
3             0.071959               0.109046                   0.051120   
4             0.035634               0.025204                   0.081354   

   sky_openness_mean  mean_segmentation_entropy  low_information_fra

                                n_lsoas
coverage_status                        
preferred_coverage                  487
acceptable_but_below_preferred        1

    LSOA21CD  IMDRank  IMDDecile  DeprivationClass  ClassIdx  seg_road_mean  \
0  E01035054    29778          9   Low Deprivation         2       0.325105   
1  E01011423     4973          2  High Deprivation         0       0.327906   
2  E01011268     6812          3  High Deprivation         0       0.275776   
3  E01033035     1856          1  High Deprivation         0       0.284325   
4  E01011339     5133          2  High Deprivation         0       0.302006   

   seg_sidewalk_mean  seg_building_mean  seg_wall_mean  seg_fence_mean  ...  \
0           0.062413           0.134347       0.014542        0.014422  ...   
1           0.032077           0.079579       0.006834        0.009990  ...   
2           0.023279           0.076686       0.004040        0.010356  ...   
3           0.060914           0.157918       0.010747        0.019481  ...   
4           0.022632           0.073024       0.003595        0.012654  ...   

    clr_bus  clr_train  clr_motorcycle  clr_bicycl

## 10. Production checks

In [15]:

final_checks = {
    "production_mode": not CONFIG.test_mode,
    "input_preflight_valid_images": int(len(image_meta)),
    "feature_rows_total": int(len(seg_img)),
    "successful_images": int(seg_img["segmentation_status"].eq("ok").sum()),
    "failed_images": int((~seg_img["segmentation_status"].eq("ok")).sum()),
    "success_rate_percent": round(100 * seg_img["segmentation_status"].eq("ok").mean(), 3),
    "lsoas_in_output": int(len(lsoa_features)),
    "spatial_blocks": int(lsoa_features["spatial_block"].nunique()),
    "spatial_folds": sorted(lsoa_features["spatial_fold"].astype(int).unique().tolist()),
    "fold_lsoa_count_cv": round(float(lsoa_cv), 6),
    "fold_image_count_cv": round(float(image_cv), 6),
    "maximum_fold_feature_abs_smd": round(float(fold_feature_balance["max_abs_smd"].max()), 6),
    "lsoas_below_minimum_coverage": int(lsoa_features["coverage_status"].eq("below_minimum").sum()),
    "image_feature_file": str(SEG_IMAGE_FEATURES_CSV),
    "lsoa_feature_file": str(SEG_LSOA_FEATURES_CSV),
    "spatial_fold_map": str(SPATIAL_FOLD_MAP_CSV),
    "spatial_fold_audit": str(SPATIAL_FOLD_AUDIT_CSV),
    "fold_feature_balance": str(SPATIAL_FOLD_FEATURE_BALANCE_CSV),
    "failure_file": str(SEG_FAILURES_CSV),
    "preflight_failure_file": str(PREFLIGHT_FAILURES_CSV),
    "qc_directory": str(QC_DIR),
}
print(json.dumps(final_checks, indent=2))

assert CONFIG.test_mode is False, "Final production run requires test_mode=False."
assert final_checks["successful_images"] > 0
assert lsoa_features[LSOA_ID_COL].is_unique
assert final_checks["spatial_folds"] == list(range(1, CONFIG.n_spatial_folds + 1))
assert lsoa_features.groupby(LSOA_ID_COL).spatial_fold.nunique().max() == 1
assert lsoa_features.groupby("spatial_block").spatial_fold.nunique().max() == 1
assert not lsoa_features[["IMDDecile", "DeprivationClass", "ClassIdx", "spatial_fold"]].isna().any().any()
assert final_checks["fold_lsoa_count_cv"] <= CONFIG.maximum_fold_size_cv
assert final_checks["fold_image_count_cv"] <= CONFIG.maximum_fold_size_cv
assert final_checks["maximum_fold_feature_abs_smd"] <= CONFIG.maximum_feature_smd

print("\nPrimary modelling instructions:")
print("1. Use exactly one LSOA per modelling row; never train the deprivation classifier on individual images.")
print("2. Hold out one complete spatial_fold at a time and train on the other four folds.")
print("3. Fit each candidate pipeline—median imputation for all models, CLR or ILR and scaling for logistic regression, class-weight options where supported, and hyperparameter tuning—only on each training partition.")
print("4. Use nested spatial CV for hyperparameter tuning; the outer held-out fold must not influence model selection.")
print("5. Report macro-F1, balanced accuracy, class-wise precision/recall/F1, confusion matrices and variability across outer folds.")
print("6. Compare grouped-7 and full-19 representations under the same frozen folds; use CLR or ILR for logistic regression and raw proportions for tree models.")
print("7. Retain coverage and low-information fields for QA; do not exclude LSOAs automatically.")
print("8. Compute Moran's I on out-of-fold residuals/errors, not on in-sample predictions, to assess remaining spatial structure.")
print("9. Preserve the exported fold map unchanged for every model so comparisons remain paired and reproducible.")


{
  "production_mode": true,
  "input_preflight_valid_images": 19372,
  "feature_rows_total": 19372,
  "successful_images": 19372,
  "failed_images": 0,
  "success_rate_percent": 100.0,
  "lsoas_in_output": 488,
  "spatial_blocks": 120,
  "spatial_folds": [
    1,
    2,
    3,
    4,
    5
  ],
  "fold_lsoa_count_cv": 0.011682,
  "fold_image_count_cv": 0.012783,
  "maximum_fold_feature_abs_smd": 0.234264,
  "lsoas_below_minimum_coverage": 0,
  "image_feature_file": "<PROJECT_DIR>/outputs_10points_metadata_once/segmentation_image_features_19class.csv",
  "lsoa_feature_file": "<PROJECT_DIR>/outputs_10points_metadata_once/segmentation_lsoa_features_19class_mean_sd.csv",
  "spatial_fold_map": "<PROJECT_DIR>/outputs_10points_metadata_once/lsoa_spatial_plus_fold_mapping.csv",
  "spatial_fold_audit": "<PROJECT_DIR>/outputs_10points_metadata_once/spatial_plus_fold_audit.csv",
  "fold_feature_balance": "<PROJECT_DIR>/outputs_10points_metadata_once/spatial_plus_fold_feature_balance.csv",
  "fai